In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
import pandas as pd
import seaborn as sns
import random
import scipy as sc
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import sys, os
sys.path.insert(0, os.path.abspath('..'))  # stormeunice is not pip-installed in this kernel
import stormeunice as eun
from numba import jit

style = eun.style
style.set_style()

random.seed(10)

# Import data

In [2]:
lon_min = -15
lon_max = 5
lat_min = 45
lat_max = 60

In [3]:
# ERA5

era5_2022 = xr.open_dataset('/gf3/predict2/AWH012_LEACH_NASTORM/DATA/ERA5/EU025/sfc/2022.nc')
era5_GUST = era5_2022.fg10.sel(time=slice('2022-02-07','2022-02-22'),latitude=slice(lat_max,lat_min),longitude=slice(lon_min, lon_max)).resample(time='3h').max().load()
era5_PRES = era5_2022.msl.sel(time=slice('2022-02-07','2022-02-22'),latitude=slice(lat_max,lat_min),longitude=slice(lon_min, lon_max)).resample(time='3h').max().load()

In [4]:
# EPS
inidates = ['2022-02-10', '2022-02-14', '2022-02-16']
experiments = ['pi', 'curr', 'incr']
eps = {}
for inidate in inidates:
    eps[inidate] = eun.data.Data.get_eps_data(experiments, inidate = inidate)

/gf3/predict2/AWH012_LEACH_NASTORM/DATA/MED-R/EXP/pi/EU025/sfc/cf/b2nq_2022-02-10.nc
/gf3/predict2/AWH012_LEACH_NASTORM/DATA/MED-R/EXP/pi/EU025/sfc/pf/b2nq_2022-02-10.nc
/gf3/predict2/AWH012_LEACH_NASTORM/DATA/MED-R/ENS/EU025/sfc/cf/1_2022-02-10.nc
/gf3/predict2/AWH012_LEACH_NASTORM/DATA/MED-R/ENS/EU025/sfc/pf/1_2022-02-10.nc
/gf3/predict2/AWH012_LEACH_NASTORM/DATA/MED-R/EXP/incr/EU025/sfc/cf/b2nr_2022-02-10.nc
/gf3/predict2/AWH012_LEACH_NASTORM/DATA/MED-R/EXP/incr/EU025/sfc/pf/b2nr_2022-02-10.nc
/gf3/predict2/AWH012_LEACH_NASTORM/DATA/MED-R/EXP/pi/EU025/sfc/cf/b2nn_2022-02-14.nc
/gf3/predict2/AWH012_LEACH_NASTORM/DATA/MED-R/EXP/pi/EU025/sfc/pf/b2nn_2022-02-14.nc
/gf3/predict2/AWH012_LEACH_NASTORM/DATA/MED-R/ENS/EU025/sfc/cf/1_2022-02-14.nc
/gf3/predict2/AWH012_LEACH_NASTORM/DATA/MED-R/ENS/EU025/sfc/pf/1_2022-02-14.nc
/gf3/predict2/AWH012_LEACH_NASTORM/DATA/MED-R/EXP/incr/EU025/sfc/cf/b2no_2022-02-14.nc
/gf3/predict2/AWH012_LEACH_NASTORM/DATA/MED-R/EXP/incr/EU025/sfc/pf/b2no_2022-02-14

In [5]:
# box for selecting max wind gusts
full_map = [lon_min, lon_max, lat_min, lat_max]
fg10_box = full_map.copy()  #[-7, 2, 49, 54]  # lon_min, lon_max, lat_min, lat_max

# Significance testing

In [6]:
# @jit(nopython=True)
def generate_boot(test, comp, n_boot=10000, sign_lev=0.9, diff_larger_0=True):
    
    '''
    Two sample bootstrap for the mean of two data sets
    '''

    x = test.values  # three dimensional array: number, lat, lon
    y = comp.values

    #Brute force
    x_sign =  np.full((x.shape[1], x.shape[2]), 0)
    for lon in range(x.shape[1]):
        for lat in range(x.shape[2]):
            x_boot = np.random.choice(x[:,lon,lat], size=(n_boot, x.shape[0]), replace=True)
            y_boot = np.random.choice(y[:,lon,lat], size=(n_boot, x.shape[0]), replace=True)

            diff = np.mean(y_boot, axis=1) - np.mean(x_boot, axis=1)
            ci_lower = np.percentile(diff, 1-sign_lev)
            ci_upper = np.percentile(diff, sign_lev)
            if diff_larger_0 & (ci_lower>0):
                x_sign[lon, lat] = 1
            elif (not diff_larger_0) & (ci_upper<0):
                x_sign[lon, lat] = 1
    return x_sign

In [7]:
# # Calculate significance for incr and pi maps with bootstraps
# sign_test = {}
# for inidate in inidates:
#     comp = eps[inidate]['curr'].sel(time = slice('2022-02-18 00', '2022-02-18 23')).fg10.max(dim = ["time"])
#     sign_test[inidate] = {}
#     for exp in ['pi', 'incr']:
#         test = eps[inidate][exp].sel(time = slice('2022-02-18 00', '2022-02-18 23')).fg10.max(dim = ["time"])
#         if exp=='pi':
#             sign_test[inidate][exp] = generate_boot(test, comp, diff_larger_0=True)
#         elif exp=='incr': 
#             sign_test[inidate][exp] = generate_boot(test, comp, diff_larger_0=False)

# Maximum wind gusts

In [11]:
# Calculate maximum gusts and minimum pressure fields across extreme composite
maximum_gusts_eps = {}
minimum_pres_eps = {}
mean_pres_eps = {}
for inidate in inidates:
    maximum_gusts_eps[inidate] = {key:value.sel(time = slice('2022-02-18 00', '2022-02-18 23')).fg10.max(dim = ["time"]).mean(dim = "number") for (key,value) in eps[inidate].items()}
    minimum_pres_eps[inidate] = {key:value.sel(time = slice('2022-02-18 00', '2022-02-18 23')).msl.min(dim = ["time"]).mean(dim = "number") for (key,value) in eps[inidate].items()}
    mean_pres_eps[inidate] = {key:value.sel(time = '2022-02-18 12').msl.mean(dim = "number") for (key,value) in eps[inidate].items()}

In [12]:
maximum_gusts_era5 = era5_GUST.sel(time = slice('2022-02-18 00', '2022-02-18 23')).max(dim = ["time"])
minimum_pres_era5 = era5_PRES.sel(time = slice('2022-02-18 00', '2022-02-18 23')).min(dim = ["time"])
mean_pres_era5 = era5_PRES.sel(time = '2022-02-18 12')

In [13]:
euroatlantic = [lon_min-13, lon_max, lat_min-5, lat_max+6]
uk = [-11, 5, 48, 60]

# Plotting

In [ ]:
# figure and map setup
projection = ccrs.PlateCarree()
p_max = 1030
p_min = 960

fg10_min = 20
fg10_max = 40

fig = plt.figure(1, figsize=(20, 11))
lead_times = ['8 days lead', '4 days lead', '2 days lead']

# type sizes are quoted at style.REF_WIDTH and scaled to this canvas, so they
# print at the same size as the other paper figures once scaled to page width
fs_title = style.scaled(style.FS_TITLE, fig)
fs_label = style.scaled(style.FS_LABEL, fig)
fs_tick = style.scaled(style.FS_TICK, fig)

panel_axes = []

# EPS data ------------------
latitude = mean_pres_eps['2022-02-10']['pi'].latitude
longitude = mean_pres_eps['2022-02-10']['pi'].longitude

for i, inidate in enumerate(inidates):
    for e, experiment in enumerate(experiments):
        p_vals = mean_pres_eps[inidate][experiment].values/100
        fg10_vals = maximum_gusts_eps[inidate][experiment].values

        ax = plt.subplot(3, 4, i+1+e*4, projection=projection)
        panel_axes.append(ax)
        ax.set_extent(uk, projection)
        ax.add_feature(cfeature.COASTLINE.with_scale('50m'), color='grey', zorder=14)
        ax.add_feature(cfeature.OCEAN, zorder=12, color='white')  # zorder > 10

        # fg10 as shading
        clevs_fg10 = np.linspace(fg10_min, fg10_max, 11)  # 17
        cf = ax.contourf(longitude, latitude, fg10_vals, clevs_fg10, cmap=style.CMAP_SEQ,
                         transform=projection, zorder=10, extend='max')

        # isobars as contours, in ink so they clear the pale end of CMAP_SEQ
        clevs_p = np.linspace(p_min, p_max, 17)
        cs = ax.contour(longitude, latitude, p_vals, clevs_p, colors=style.CONTOUR_DARK,
                        linewidths=0.8, transform=projection, zorder=16)
        plt.clabel(cs, fmt='%d', fontsize=fs_tick)
        ax.set_title(lead_times[i] if experiment == 'pi' else '',
                     fontsize=fs_title, fontweight='bold')

        # # hatching for significance
        # if experiment in ['pi', 'incr']:
        #     sign = sign_test[inidate][experiment]
        #     ax.contourf(longitude, latitude, sign, alpha=0, hatches=[None, style.HATCH], n_levels=2, zorder=11)

# ERA5 data ----------------------
latitude = mean_pres_era5.latitude
longitude = mean_pres_era5.longitude

p_vals = mean_pres_era5.values/100
fg10_vals = maximum_gusts_era5.values
ax_era5 = plt.subplot(3, 4, 8, projection=projection)
ax_era5.set_extent(uk, projection)
ax_era5.add_feature(cfeature.COASTLINE.with_scale('50m'), color='grey', zorder=14)
ax_era5.add_feature(cfeature.OCEAN, zorder=12, color='white')  # zorder > 10

# fg10 as shading
clevs_fg10 = np.linspace(fg10_min, fg10_max, 11)  # 17
cf = ax_era5.contourf(longitude, latitude, fg10_vals, clevs_fg10, cmap=style.CMAP_SEQ,
                      transform=projection, zorder=10, extend='max')

# isobars as contours
clevs_p = np.linspace(p_min, p_max, 17)
cs = ax_era5.contour(longitude, latitude, p_vals, clevs_p, colors=style.CONTOUR_DARK,
                     linewidths=0.8, transform=projection, zorder=16)
plt.clabel(cs, fmt='%d', fontsize=fs_tick)
ax_era5.set_title('ERA5', fontsize=fs_title, fontweight='bold')

# panel labels a, b, c, ... in the order the panels are drawn
style.panel_labels(panel_axes + [ax_era5])

# Other figure settings -----------------
ax = plt.subplot(3, 4, 4)
ax.axis('off')  # removes subplot frame
cax = ax.inset_axes([0.47, 0.02, 0.1, 0.95])  # creates inset, [x0,y0, width, height]
cbar = fig.colorbar(cf, cax=cax, extend='max', shrink=0.8)
cbar.set_label(label='Wind gusts (m s$^{-1}$)', size=fs_label)
cbar.ax.tick_params(labelsize=fs_tick)

# row headers, each in its experiment's colour
for y, experiment in zip([0.82, 0.48, 0.15], experiments):
    plt.figtext(0.01, y, style.experiment_label(experiment), rotation='vertical',
                fontsize=fs_label, fontweight='bold',
                color=style.experiment_color(experiment))
plt.tight_layout()

style.save(fig, 'PAPER-SUPP1_max_gusts_allmembers', directory='figures')